# Đánh giá open-weight cho cAI (SP1b)

1. **Add Input** → dataset PRIVATE vừa upload (thư mục do `python -m tools.model_eval kaggle-ghi` tạo: `requests.jsonl`, `meta.json`, `chay_kaggle.py`).
2. **Settings** → Accelerator **GPU T4 x2**, **Internet on**.
3. **Run all**. Xong tải `/kaggle/working/outputs.zip`, giải nén vào `docs/model-eval/<ngày>-kaggle/outputs` rồi chạy `python -m tools.model_eval kaggle-cham`.

**Bản sửa:** cell kế tiếp ghi `chay_kaggle.py` đã sửa (thêm `--enforce-eager`, `--disable-custom-all-reduce` cho T4; in đúng dòng lỗi gốc của vLLM) vào `/kaggle/working` và được ưu tiên hơn bản trong dataset — dùng lại dataset cũ được, không cần upload lại.

Dữ liệu chỉ là nội quy TỔNG HỢP + câu hỏi eval. Không nhập key/token nào vào notebook này.

In [1]:
%%writefile /kaggle/working/chay_kaggle.py
"""Chạy prompt eval SP1 trên Kaggle (T4×2) bằng vLLM, ghi output để phát lại qua E2E ở local (SP1b).

File này nằm TRONG dataset Kaggle và chạy độc lập: chỉ dùng thư viện chuẩn, không cần repo cai-workspace.
vLLM được gọi qua CLI `vllm serve` + HTTP, không import SDK nào (luật "SDK model chỉ trong
harness/modelgateway"). Ở local, `backend="stub"` thay vLLM bằng server giả để thử trọn đường trước khi upload.
Spec: docs/superpowers/specs/2026-09-26-sp1b-kaggle-open-weight-design.md
"""

from __future__ import annotations

import http.client
import json
import os
import re
import signal
import subprocess
import sys
import threading
import time
from collections.abc import Callable
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import IO, Any

#: Bản vLLM đã có người kiểm trên Kaggle T4×2 (CUDA 12.8) — github.com/kaggle-vllm/kaggle-vllm. Qwen3 cần ≥ 0.8.5.
VLLM_PHIEN_BAN = "0.18.1"
HAN_REQUEST_GIAY = 300.0
CONG_VLLM = 8000
#: T4 không có bf16 → half. 2 card → TP 2. 16k context đủ cho ~6k token bằng chứng + suy luận.
#: `--enforce-eager` + `--disable-custom-all-reduce`: đúng cấu hình kaggle-vllm đã kiểm trên T4 (SM75,
#: 2 card không NVLink) — bỏ torch.compile/CUDA graph và custom all-reduce, chậm hơn nhưng khởi động được.
_SERVER_CHUNG = [
    "--dtype",
    "half",
    "--tensor-parallel-size",
    "2",
    "--max-model-len",
    "16384",
    "--gpu-memory-utilization",
    "0.90",
    "--reasoning-parser",
    "qwen3",
    "--enforce-eager",
    "--disable-custom-all-reduce",
]
CAU_HINH: list[dict[str, Any]] = [
    {
        "id": "kaggle-qwen3-14b-awq",
        "model": "Qwen/Qwen3-14B-AWQ",
        "server": ["--quantization", "awq", *_SERVER_CHUNG],
        "request": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    {
        "id": "kaggle-qwen3-14b-awq-think",
        "model": "Qwen/Qwen3-14B-AWQ",
        "server": ["--quantization", "awq", *_SERVER_CHUNG],
        "request": {"chat_template_kwargs": {"enable_thinking": True}},
    },
    {
        "id": "kaggle-qwen3-8b",
        "model": "Qwen/Qwen3-8B",
        "server": list(_SERVER_CHUNG),
        "request": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    {
        "id": "kaggle-qwen3-8b-think",
        "model": "Qwen/Qwen3-8B",
        "server": list(_SERVER_CHUNG),
        "request": {"chat_template_kwargs": {"enable_thinking": True}},
    },
]

XuLy = Callable[[str, str, dict[str, Any]], tuple[int, dict[str, Any]]]
#: Thẻ rào bằng chứng do `harness/executor/tra_loi.py::dung_tin_nhan` dựng (chép — file này phải độc lập).
_MUC = re.compile(r'<muc item_id="([^"]+)">\n(.*?)\n</muc>', re.S)
#: process_request_thread chạy handle_error trên đúng luồng đã gọi handler — thread-local này mang
#: theo (phương thức, đường dẫn) của request đang xử lý để in một dòng chẩn đoán không lẫn luồng khác.
_yeu_cau_hien_tai = threading.local()
_LOI_DUONG_TRUYEN = (ConnectionAbortedError, BrokenPipeError, ConnectionResetError)
#: Dòng log vllm mang nguyên nhân gốc. Lỗi thật nằm ở tiến trình EngineCore/Worker, cao hơn nhiều so với
#: traceback APIServer ở cuối log (APIServer chỉ báo lại "engine khởi tạo lỗi").
_DONG_LOI_LOG = re.compile(r"Error|Exception|out of memory|CUDA error|NCCL")
#: Giới hạn độ dài lỗi ghi vào moi_truong.json (bản in ra màn hình thì đầy đủ).
_DAI_LOI_GHI = 2000


class _MayChu(ThreadingHTTPServer):
    """Lỗi do đường truyền (client đã bỏ cuộc quá hạn, đóng socket giữa chừng) thì im lặng — đó là
    kết quả mong đợi, không phải lỗi cần báo. Lỗi khác thì in đúng MỘT dòng: tên loại lỗi + đường dẫn
    request, không traceback, không thông điệp lỗi (thông điệp có thể mang nội dung request/prompt)."""

    def handle_error(self, *_: Any) -> None:
        loi = sys.exc_info()[1]
        if isinstance(loi, _LOI_DUONG_TRUYEN):
            return
        ten = type(loi).__name__ if loi is not None else "LoiKhongRo"
        phuong_thuc = getattr(_yeu_cau_hien_tai, "phuong_thuc", "?")
        duong = getattr(_yeu_cau_hien_tai, "duong", "?")
        print(f"[may_chu] lỗi {ten} khi xử lý {phuong_thuc} {duong}", file=sys.stderr, flush=True)


def dung_may_chu(xu_ly: XuLy, host: str = "127.0.0.1", cong: int = 0) -> ThreadingHTTPServer:
    """Server JSON nhỏ trên thread daemon. Không log request: thân request chứa prompt."""

    class _XuLy(BaseHTTPRequestHandler):
        def _chay(self, phuong_thuc: str) -> None:
            body: dict[str, Any] = {}
            if phuong_thuc == "POST":
                try:
                    body = json.loads(
                        self.rfile.read(int(self.headers.get("content-length") or 0)) or b"{}"
                    )
                except ValueError:
                    self._tra(400, {"error": {"message": "thân không phải JSON"}})
                    return
            _yeu_cau_hien_tai.phuong_thuc = phuong_thuc
            _yeu_cau_hien_tai.duong = self.path
            ma, tra = xu_ly(phuong_thuc, self.path, body)
            self._tra(ma, tra)

        def do_GET(self) -> None:
            self._chay("GET")

        def do_POST(self) -> None:
            self._chay("POST")

        def _tra(self, ma: int, tra: dict[str, Any]) -> None:
            du_lieu = json.dumps(tra, ensure_ascii=False).encode("utf-8")
            self.send_response(ma)
            self.send_header("content-type", "application/json")
            self.send_header("content-length", str(len(du_lieu)))
            self.end_headers()
            self.wfile.write(du_lieu)

        def log_message(self, *_: Any) -> None:
            return

    may = _MayChu((host, cong), _XuLy)
    threading.Thread(target=may.serve_forever, daemon=True).start()
    return may


def tra_loi_gia(body: dict[str, Any]) -> dict[str, Any]:
    """Câu trả lời hợp lệ theo LUOC_DO_TRA_LOI: nguyên văn mục bằng chứng đầu tiên (như MockTrichNguon)."""
    he_thong = next(
        (str(m.get("content", "")) for m in body.get("messages", []) if m.get("role") == "system"),
        "",
    )
    muc = _MUC.search(he_thong)
    if muc is None:
        return {"answer": "Chưa đủ căn cứ để trả lời.", "citations": [], "khong_du_can_cu": True}
    return {
        "answer": muc.group(2).replace("&lt;", "<"),
        "citations": [muc.group(1)],
        "khong_du_can_cu": False,
    }


def phan_hoi_chat(noi_dung: str, model: str, usage: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": "chatcmpl-sp1b",
        "object": "chat.completion",
        "model": model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": noi_dung},
                "finish_reason": "stop",
            }
        ],
        "usage": dict(usage),
    }


def _xu_ly_gia(phuong_thuc: str, duong: str, body: dict[str, Any]) -> tuple[int, dict[str, Any]]:
    """Server giả thay vLLM khi chạy thử ở local (backend="stub")."""
    if phuong_thuc == "GET":
        return (200, {"data": []}) if duong == "/v1/models" else (404, {})
    noi_dung = json.dumps(tra_loi_gia(body), ensure_ascii=False)
    tong = sum(len(str(m.get("content", ""))) for m in body.get("messages", []))
    usage = {"prompt_tokens": max(1, tong // 4), "completion_tokens": max(1, len(noi_dung) // 4)}
    return 200, phan_hoi_chat(noi_dung, str(body.get("model") or "gia"), usage)


def doc_dataset(thu_muc: Path) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    tho = (thu_muc / "requests.jsonl").read_text(encoding="utf-8")
    requests = [json.loads(d) for d in tho.splitlines() if d.strip()]
    return requests, json.loads((thu_muc / "meta.json").read_text(encoding="utf-8"))


def than_request(body: dict[str, Any], cau_hinh: dict[str, Any]) -> dict[str, Any]:
    """Đúng thân đã ghi, thay `model`, trộn tham số request của cấu hình. Không thêm temperature/max_tokens."""
    return {**body, **cau_hinh["request"], "model": cau_hinh["model"]}


def gui(host: str, cong: int, than: dict[str, Any], han_giay: float) -> dict[str, Any]:
    """Một request chat/completions. Lỗi chỉ ghi tên lỗi — không bao giờ chép nội dung."""
    bat_dau = time.monotonic()
    ket: dict[str, Any] = {
        "ok": False,
        "content": None,
        "usage": {},
        "model": than.get("model"),
        "finish_reason": None,
        "loi": "",
    }
    conn = http.client.HTTPConnection(host, cong, timeout=han_giay)
    try:
        conn.request(
            "POST",
            "/v1/chat/completions",
            body=json.dumps(than, ensure_ascii=False).encode("utf-8"),
            headers={"content-type": "application/json"},
        )
        r = conn.getresponse()
        tho = r.read()
        if r.status != 200:
            ket["loi"] = f"HTTP {r.status}"
        else:
            du_lieu = json.loads(tho)
            lua_chon = du_lieu["choices"][0]
            noi_dung = lua_chon["message"]["content"]
            ket["finish_reason"] = lua_chon.get("finish_reason")
            ket["content"] = noi_dung
            ket["usage"] = du_lieu.get("usage") or {}
            ket["model"] = du_lieu.get("model") or than.get("model")
            if noi_dung:
                ket["ok"] = True
            else:
                ket["loi"] = f"content rỗng (finish_reason={ket['finish_reason']})"
    except TimeoutError:
        ket["loi"] = f"quá hạn {han_giay:g}s"
    except (OSError, ValueError, KeyError, IndexError, TypeError, http.client.HTTPException) as loi:
        ket["loi"] = type(loi).__name__
    finally:
        conn.close()
    ket["ms"] = int((time.monotonic() - bat_dau) * 1000)
    return ket


def chay_ung_vien(
    host: str,
    cong: int,
    requests: list[dict[str, Any]],
    cau_hinh: dict[str, Any],
    han_giay: float = HAN_REQUEST_GIAY,
) -> list[dict[str, Any]]:
    dong: list[dict[str, Any]] = []
    for r in requests:
        ket = {"khoa": r["khoa"], **gui(host, cong, than_request(r["body"], cau_hinh), han_giay)}
        dong.append(ket)
        print(
            f"[kaggle] {cau_hinh['id']} {r['khoa'][:8]} {'ok' if ket['ok'] else 'loi'} "
            f"{ket['ms']} ms",
            flush=True,
        )
    return dong


def ghi_jsonl(duong: Path, dong: list[dict[str, Any]]) -> None:
    duong.parent.mkdir(parents=True, exist_ok=True)
    duong.write_text(
        "".join(json.dumps(d, ensure_ascii=False) + "\n" for d in dong),
        encoding="utf-8",
        newline="\n",
    )


def _duoi_log(log: Path | None, so_dong: int = 40, so_dong_loi: int = 30) -> str:
    """Trích log vllm để chẩn đoán (log là output của vllm, không phải prompt): trước hết tối đa
    `so_dong_loi` dòng lỗi không thuộc APIServer (nguyên nhân gốc), sau đó ~`so_dong` dòng cuối."""
    if log is None or not log.is_file():
        return ""
    dong = log.read_text(encoding="utf-8", errors="replace").splitlines()
    loi = [d for d in dong if _DONG_LOI_LOG.search(d) and "APIServer" not in d][:so_dong_loi]
    phan = []
    if loi:
        phan.append("--- dòng lỗi (EngineCore/Worker) ---\n" + "\n".join(loi))
    if dong:
        phan.append("--- cuối log ---\n" + "\n".join(dong[-so_dong:]))
    return "\n" + "\n".join(phan) if phan else ""


def cho_san_sang(
    host: str,
    cong: int,
    han_giay: float = 900.0,
    ngu: Callable[[float], None] = time.sleep,
    proc: subprocess.Popen[bytes] | None = None,
    log: Path | None = None,
) -> None:
    het = time.monotonic() + han_giay
    while time.monotonic() < het:
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(
                f"vllm thoát sớm (mã {proc.returncode}) trước khi sẵn sàng{_duoi_log(log)}"
            )
        try:
            conn = http.client.HTTPConnection(host, cong, timeout=5)
            conn.request("GET", "/v1/models")
            san_sang = conn.getresponse().status == 200
            conn.close()
            if san_sang:
                return
        except OSError:
            pass
        ngu(5)
    raise RuntimeError(f"server model không sẵn sàng sau {han_giay:g}s{_duoi_log(log)}")


def cai_vllm(phien_ban: str = VLLM_PHIEN_BAN) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"vllm=={phien_ban}"])


def khoi_dong_vllm(
    cau_hinh: dict[str, Any], cong: int, log: Path
) -> tuple[subprocess.Popen[bytes], IO[bytes]]:
    """Trả cả handle log để gọi nơi đã mở nó đóng lại sau khi tiến trình kết thúc.

    `start_new_session=True` (POSIX) làm proc thành process-group leader, để `_dung_vllm` dừng được
    cả worker con vllm spawn ra, không chỉ tiến trình cha. Windows không có khái niệm này — bỏ qua
    (harmless), `_dung_vllm` dùng terminate/kill thường ở đó.
    """
    log.parent.mkdir(parents=True, exist_ok=True)
    tep = log.open("wb")
    kwargs: dict[str, Any] = {"start_new_session": True} if sys.platform != "win32" else {}
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "vllm.entrypoints.cli.main",
            "serve",
            cau_hinh["model"],
            "--port",
            str(cong),
            *cau_hinh["server"],
        ],
        stdout=tep,
        stderr=subprocess.STDOUT,
        **kwargs,
    )
    return proc, tep


def _dung_vllm(proc: subprocess.Popen[bytes]) -> None:
    """Dừng cả process group (POSIX, `os.killpg` — vllm spawn worker con); Windows: terminate/kill."""
    if sys.platform != "win32":
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()
    else:
        proc.terminate()
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()


def moi_truong(backend: str) -> dict[str, Any]:
    def _lenh(*lenh: str) -> str:
        try:
            return subprocess.run(
                list(lenh), capture_output=True, text=True, timeout=60, check=False
            ).stdout.strip()
        except (OSError, subprocess.SubprocessError):
            return ""

    return {
        "backend": backend,
        "python": sys.version.split()[0],
        "vllm": _lenh("vllm", "--version") if backend == "vllm" else "",
        "gpu": _lenh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"),
    }


def _ghi_moi_truong(ra: Path, ket: dict[str, Any]) -> None:
    (ra / "moi_truong.json").write_text(
        json.dumps(ket, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def chay_tat_ca(
    dataset: Path,
    ra: Path,
    *,
    backend: str = "vllm",
    cau_hinh: list[dict[str, Any]] | None = None,
    han_giay: float = HAN_REQUEST_GIAY,
    han_san_sang_giay: float = 1800.0,
) -> dict[str, Any]:
    """Mỗi model một server (vLLM hoặc giả), chạy lần lượt các cấu hình của model đó, ghi `ra/<id>.jsonl`.

    Một nhóm (model) lỗi lúc dựng/chờ server (vllm chết sớm, timeout sẵn sàng...) không làm dừng cả lượt:
    lỗi được ghi vào từng ứng viên của nhóm đó và lượt tiếp tục với model kế tiếp. `moi_truong.json` được
    ghi lại sau mỗi ứng viên và trong `finally` ngoài cùng, nên file này luôn tồn tại kể cả khi lượt bị
    ngắt giữa chừng.
    """
    if backend not in ("vllm", "stub"):
        raise ValueError(f"backend {backend!r} — chỉ nhận 'vllm' hoặc 'stub'")
    cau_hinh = CAU_HINH if cau_hinh is None else cau_hinh
    requests, meta = doc_dataset(dataset)
    ra.mkdir(parents=True, exist_ok=True)
    ket: dict[str, Any] = {
        **moi_truong(backend),
        "dataset": {
            k: meta.get(k) for k in ("git_sha", "content_hash", "prompt_version", "so_request")
        },
        "ung_vien": {},
    }
    try:
        for model in dict.fromkeys(c["model"] for c in cau_hinh):
            nhom = [c for c in cau_hinh if c["model"] == model]
            may: ThreadingHTTPServer | None = None
            proc: subprocess.Popen[bytes] | None = None
            log_tep: IO[bytes] | None = None
            log_duong = ra / f"vllm-{nhom[0]['id']}.log"
            try:
                if backend == "stub":
                    may = dung_may_chu(_xu_ly_gia)
                    cong = int(may.server_address[1])
                else:
                    cong = CONG_VLLM
                    proc, log_tep = khoi_dong_vllm(nhom[0], cong, log_duong)
                cho_san_sang("127.0.0.1", cong, han_san_sang_giay, proc=proc, log=log_duong)
                for c in nhom:
                    dong = chay_ung_vien("127.0.0.1", cong, requests, c, han_giay)
                    ghi_jsonl(ra / f"{c['id']}.jsonl", dong)
                    ket["ung_vien"][c["id"]] = {
                        "model": c["model"],
                        "server": c["server"],
                        "tham_so_request": c["request"],
                        "ok": sum(1 for d in dong if d["ok"]),
                        "tong": len(dong),
                    }
                    print(
                        f"[kaggle] {c['id']}: ok {ket['ung_vien'][c['id']]['ok']}/{len(dong)}",
                        flush=True,
                    )
                    _ghi_moi_truong(ra, ket)
            except Exception as loi:
                thong_diep = f"{type(loi).__name__}: {loi}"
                print(f"[kaggle] {model} lỗi, bỏ qua model này: {thong_diep}", flush=True)
                thong_diep = thong_diep[:_DAI_LOI_GHI]
                for c in nhom:
                    ket["ung_vien"][c["id"]] = {
                        "model": c["model"],
                        "server": c["server"],
                        "tham_so_request": c["request"],
                        "ok": 0,
                        "tong": len(requests),
                        "loi": thong_diep,
                    }
                _ghi_moi_truong(ra, ket)
            finally:
                if may is not None:
                    may.shutdown()
                if proc is not None:
                    _dung_vllm(proc)
                if log_tep is not None:
                    log_tep.close()
    finally:
        _ghi_moi_truong(ra, ket)
    return ket


def tom_tat(ra: Path) -> str:
    """Một dòng mỗi ứng viên: ok/tổng, p50 ms."""
    dong_ra = []
    for tep in sorted(ra.glob("*.jsonl")):
        dong = [json.loads(x) for x in tep.read_text(encoding="utf-8").splitlines() if x.strip()]
        ms = sorted(d["ms"] for d in dong if d["ok"])
        p50 = ms[len(ms) // 2] if ms else None
        dong_ra.append(
            f"{tep.stem}: ok {sum(1 for d in dong if d['ok'])}/{len(dong)}, p50 {p50} ms"
        )
    return "\n".join(dong_ra)


Writing /kaggle/working/chay_kaggle.py


In [2]:
import glob
import json
import os
import pathlib
import sys

BACKEND = os.environ.get("KAGGLE_BACKEND", "vllm")
if "KAGGLE_DATASET" in os.environ:
    DATASET = pathlib.Path(os.environ["KAGGLE_DATASET"])
else:
    ung_vien = sorted(glob.glob("/kaggle/input/**/requests.jsonl", recursive=True))
    if not ung_vien:
        raise RuntimeError(
            "chưa thấy requests.jsonl dưới /kaggle/input -- chưa add dataset (Add Input) chứa "
            "requests.jsonl/meta.json/chay_kaggle.py do `kaggle-ghi` tạo"
        )
    if len(ung_vien) > 1:
        raise RuntimeError(
            "thấy nhiều requests.jsonl dưới /kaggle/input, không rõ dùng cái nào -- chỉ attach một "
            f"dataset cho notebook này: {ung_vien}"
        )
    DATASET = pathlib.Path(ung_vien[0]).parent
OUT = pathlib.Path(os.environ.get("KAGGLE_OUT", "/kaggle/working/outputs"))
sys.path.insert(0, str(DATASET))
# bản chay_kaggle.py đã sửa (cell c1b) đứng trước bản trong dataset
sys.path.insert(0, "/kaggle/working")
import chay_kaggle as ck  # noqa: E402 -- phai insert sys.path truoc khi import

print("chay_kaggle:", ck.__file__)
print("backend:", BACKEND, "| dataset:", DATASET, "| out:", OUT, "| vllm:", ck.VLLM_PHIEN_BAN)

chay_kaggle: /kaggle/working/chay_kaggle.py
backend: vllm | dataset: /kaggle/input/datasets/tuantruong2004/vgent-dataset-v2 | out: /kaggle/working/outputs | vllm: 0.18.1


In [3]:
if BACKEND == "vllm":
    ck.cai_vllm(ck.VLLM_PHIEN_BAN)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.

In [4]:
if BACKEND == "vllm":
    import torch

    print(
        "torch:",
        torch.__version__,
        "| cuda:",
        torch.cuda.is_available(),
        "| số GPU:",
        torch.cuda.device_count(),
    )

torch: 2.10.0+cu128 | cuda: True | số GPU: 2


In [5]:
moi_truong = ck.chay_tat_ca(DATASET, OUT, backend=BACKEND)
print(
    json.dumps(
        {k: v for k, v in moi_truong.items() if k != "ung_vien"}, ensure_ascii=False, indent=2
    )
)

[kaggle] Qwen/Qwen3-14B-AWQ lỗi, bỏ qua model này: RuntimeError: vllm thoát sớm (mã 1) trước khi sẵn sàng
--- dòng lỗi (EngineCore/Worker) ---
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932]     raise CalledProcessError(retcode, process.args,
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932] subprocess.CalledProcessError: Command '['ninja', '-v', '-C', '/root/.cache/flashinfer/0.6.6/75/cached_ops/batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_f16_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False', '-f', '/root/.cache/flashinfer/0.6.6/75/cached_ops/batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_f16_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False/build.ninja']' returned non-zero exit status 1.
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932]     raise RuntimeError(msg) from e
(Worker_TP1 pid=21

In [6]:
# Chẩn đoán: dòng lỗi gốc trong log vllm của từng model (chỉ output vllm, không có prompt)
import re

for log in sorted(OUT.glob("vllm-*.log")):
    dong = log.read_text(encoding="utf-8", errors="replace").splitlines()
    loi = [d for d in dong if re.search(r"Error|Exception|out of memory|CUDA error|NCCL", d) and "APIServer" not in d]
    print(f"=== {log.name}: {len(loi)} dòng lỗi")
    print("\n".join(loi[:30]))

=== vllm-kaggle-qwen3-14b-awq.log: 20 dòng lỗi
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932]     raise CalledProcessError(retcode, process.args,
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932] subprocess.CalledProcessError: Command '['ninja', '-v', '-C', '/root/.cache/flashinfer/0.6.6/75/cached_ops/batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_f16_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False', '-f', '/root/.cache/flashinfer/0.6.6/75/cached_ops/batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_f16_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False/build.ninja']' returned non-zero exit status 1.
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932]     raise RuntimeError(msg) from e
(Worker_TP1 pid=214) ERROR 09-27 04:49:39 [multiproc_executor.py:932] RuntimeError: Ninja build failed. Ninja outp

In [7]:
import shutil

print(ck.tom_tat(OUT))
shutil.make_archive(str(OUT), "zip", OUT)
print("→", str(OUT) + ".zip")


→ /kaggle/working/outputs.zip
